# FinGuard AI — PaySim Fraud Detection
### TCN + BiLSTM + Multi-Head Attention
---

# PaySim Dataset — TCN + BiLSTM + Attention

## 0. Installs & Imports

In [ ]:
!unzip -q /content/PS_20174392719_1491204439457_log.csv.zip

replace PS_20174392719_1491204439457_log.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [ ]:
!pip install faker --quiet

import os, random, warnings, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, precision_recall_curve,
    average_precision_score, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 1. Load PaySim (or clean Faker fallback)

### Leakage fix applied here
Features `newbalanceOrig`, `newbalanceDest`, `error_balance_*` are REMOVED —
they are computed AFTER the transaction completes, so they are not available
at detection time and directly encode the fraud outcome.
Only pre-transaction features are kept.

In [ ]:
PAYSIM_CSV = '/content/PS_20174392719_1491204439457_log.csv'

def generate_clean_faker(n_rows=200_000, seed=42):
    """
    Faker dataset with realistic noise in fraud patterns
    so no single feature perfectly separates classes.
    """
    from faker import Faker
    Faker.seed(seed); np.random.seed(seed); random.seed(seed)
    fake = Faker()
    TYPES     = ['CASH_IN','CASH_OUT','DEBIT','PAYMENT','TRANSFER']
    FRAUD_PROB = {'CASH_IN':0,'CASH_OUT':0.004,'DEBIT':0,'PAYMENT':0,'TRANSFER':0.004}
    records = []
    for step in range(1, 745):
        hour = step % 24
        mult = 2.0 if (8<=hour<=10 or 17<=hour<=20) else 1.0
        n    = int(np.random.poisson(n_rows/744*mult))
        for _ in range(n):
            t      = random.choices(TYPES, weights=[.25,.35,.05,.25,.10])[0]
            amount = round(min(float(np.random.lognormal(7,2)),9_999_999),2)
            ob     = round(random.uniform(100,500_000),2)   # always >0
            db     = round(random.uniform(0,500_000),2)
            is_fraud = int(random.random() < FRAUD_PROB[t])
            if is_fraud:
                # NOISE: drain 70-100% of balance, not exactly 100%
                drain_pct = random.uniform(0.70, 1.00)
                amount = round(ob * drain_pct, 2)
            nob = max(0, ob - amount) if t in ('CASH_OUT','TRANSFER','DEBIT','PAYMENT') else ob+amount
            ndb = db + amount if is_fraud else db
            records.append({'step':step,'type':t,'amount':amount,
                'nameOrig':fake.bothify('C##########'),
                'oldbalanceOrg':ob,'newbalanceOrig':round(nob,2),
                'nameDest':fake.bothify('C##########'),
                'oldbalanceDest':db,'newbalanceDest':round(ndb,2),
                'isFraud':is_fraud,'isFlaggedFraud':int(is_fraud and amount>200_000)})
    return pd.DataFrame(records).iloc[:n_rows].reset_index(drop=True)

if os.path.exists(PAYSIM_CSV):
    print("Loading real PaySim …")
    df = pd.read_csv(PAYSIM_CSV)
else:
    print("Generating clean Faker dataset …")
    df = generate_clean_faker(200_000, seed=42)

print(df.shape, "| Fraud ratio:", df['isFraud'].mean().round(5))

Loading real PaySim …
(6362620, 11) | Fraud ratio: 0.00129


## 2. Feature Engineering — leakage-free

In [ ]:
df = df.drop(columns=['nameOrig','nameDest'], errors='ignore')
df['type'] = df['type'].str.replace('-','_').str.upper()
df = pd.get_dummies(df, columns=['type'], prefix='type')
type_cols = [c for c in df.columns if c.startswith('type_')]
df[type_cols] = df[type_cols].astype(np.float32)

# ── ONLY pre-transaction features ─────────────────────────────────────────────
# newbalanceOrig / newbalanceDest / error_balance_* are POST-transaction → REMOVED
EPS = 1e-9
df['amount_to_orig_bal_ratio'] = (df['amount'] / (df['oldbalanceOrg'] + EPS)).clip(upper=df['amount'].quantile(.999))
df['amount_to_dest_bal_ratio'] = (df['amount'] / (df['oldbalanceDest'] + EPS)).clip(upper=df['amount'].quantile(.999))
# oldbalanceOrg == 0 flag: fraud often empties accounts
df['orig_bal_zero'] = (df['oldbalanceOrg'] == 0).astype(np.float32)

df['hour_sin'] = np.sin(2*np.pi*(df['step']%24)/24).astype(np.float32)
df['hour_cos'] = np.cos(2*np.pi*(df['step']%24)/24).astype(np.float32)
df['dow_sin']  = np.sin(2*np.pi*((df['step']//24)%7)/7).astype(np.float32)
df['dow_cos']  = np.cos(2*np.pi*((df['step']//24)%7)/7).astype(np.float32)

for col in ['amount','oldbalanceOrg','oldbalanceDest',
            'amount_to_orig_bal_ratio','amount_to_dest_bal_ratio']:
    cap = df[col].quantile(0.999)
    df[col] = np.log1p(df[col].clip(upper=cap)).astype(np.float32)

df = df.sort_values('step').reset_index(drop=True)

# ── Feature list (NO post-transaction columns) ─────────────────────────────────
FEATURE_COLS = (
    ['amount','oldbalanceOrg','oldbalanceDest',
     'amount_to_orig_bal_ratio','amount_to_dest_bal_ratio','orig_bal_zero']
    + ['hour_sin','hour_cos','dow_sin','dow_cos']
    + type_cols
)
TARGET = 'isFraud'
N_FEATURES = len(FEATURE_COLS)
print(f"Features ({N_FEATURES}):", FEATURE_COLS)
print("Fraud ratio:", df[TARGET].mean().round(5))

Features (15): ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'amount_to_orig_bal_ratio', 'amount_to_dest_bal_ratio', 'orig_bal_zero', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']
Fraud ratio: 0.00129


## 3. Chronological split + Dataset (WeightedRandomSampler)

In [ ]:
SEQ_LEN = 24
N         = len(df)
train_end = int(0.70 * N)
val_end   = int(0.85 * N)

df_train = df.iloc[:train_end].reset_index(drop=True)
df_val   = df.iloc[train_end:val_end].reset_index(drop=True)
df_test  = df.iloc[val_end:].reset_index(drop=True)

print(f"Train {len(df_train):,} | Val {len(df_val):,} | Test {len(df_test):,}")
print(f"Fraud rate — Train: {df_train[TARGET].mean():.4%} | Test: {df_test[TARGET].mean():.4%}")

class FraudSeqDataset(Dataset):
    """Sliding window; NO oversampling — use WeightedRandomSampler instead."""
    def __init__(self, df, feature_cols, target, seq_len):
        X = df[feature_cols].values.astype(np.float32)
        y = df[target].values.astype(np.float32)
        n = len(X) - seq_len + 1
        self.X = np.lib.stride_tricks.sliding_window_view(
            X, (seq_len, X.shape[1])).reshape(n, seq_len, X.shape[1])
        self.y = y[seq_len - 1:]
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i])

BATCH_SIZE = 512
train_ds = FraudSeqDataset(df_train, FEATURE_COLS, TARGET, SEQ_LEN)
val_ds   = FraudSeqDataset(df_val,   FEATURE_COLS, TARGET, SEQ_LEN)
test_ds  = FraudSeqDataset(df_test,  FEATURE_COLS, TARGET, SEQ_LEN)

print(f"Train windows {len(train_ds):,} | fraud ratio {train_ds.y.mean():.4%}")

# WeightedRandomSampler: keeps ALL windows, upsamples fraud at batch level
n_fraud  = int(train_ds.y.sum())
n_normal = len(train_ds) - n_fraud
weights  = np.where(train_ds.y == 1, n_normal/n_fraud, 1.0)
sampler  = WeightedRandomSampler(weights, num_samples=len(train_ds), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders ready ✓")

Train 4,453,834 | Val 954,393 | Test 954,393
Fraud rate — Train: 0.0818% | Test: 0.4200%
Train windows 4,453,811 | fraud ratio 0.0818%
DataLoaders ready ✓


## 4. Model: TCN + BiLSTM + Multi-Head Attention

In [ ]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, dilation=1):
        super().__init__()
        pad = (kernel-1)*dilation
        self.conv1 = nn.Conv1d(in_ch,  out_ch, kernel, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=pad, dilation=dilation)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1) if in_ch!=out_ch else nn.Identity()
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.drop  = nn.Dropout(0.1)
        self._pad  = pad
    def forward(self, x):
        r = self.skip(x)
        o = F.gelu(self.bn1(self.conv1(x)[...,:-self._pad] if self._pad else self.conv1(x)))
        o = self.drop(o)
        o = F.gelu(self.bn2(self.conv2(o)[...,:-self._pad] if self._pad else self.conv2(o)))
        return o + r

class FraudDetector(nn.Module):
    def __init__(self, n_features, hidden=128, n_heads=4, dropout=0.3):
        super().__init__()
        self.tcn = nn.Sequential(
            TCNBlock(n_features, 64,  kernel=3, dilation=1),
            TCNBlock(64,        128, kernel=3, dilation=2),
            TCNBlock(128,       128, kernel=3, dilation=4),
        )
        self.bilstm   = nn.LSTM(128, hidden//2, num_layers=2, batch_first=True,
                                bidirectional=True, dropout=dropout)
        self.attn     = nn.MultiheadAttention(hidden, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm= nn.LayerNorm(hidden)
        self.head     = nn.Sequential(
            nn.Linear(hidden,64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,32),    nn.GELU(), nn.Dropout(dropout/2),
            nn.Linear(32,1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x):
        h = self.tcn(x.permute(0,2,1)).permute(0,2,1)
        h,_ = self.bilstm(h)
        a,_ = self.attn(h,h,h)
        h   = self.attn_norm(h+a)
        pooled = (h.mean(1) + h[:,-1,:]) / 2
        return self.head(pooled).squeeze(-1)

model = FraudDetector(N_FEATURES, 128, 4, 0.3).to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Params: 473,793


## 5. Focal Loss + Training

In [ ]:
def focal_loss(logits, targets, gamma=2.0, alpha=0.75):
    p    = torch.sigmoid(logits)
    p_t  = torch.where(targets==1, p, 1-p)
    a_t  = torch.where(targets==1, torch.full_like(targets,alpha),
                                   torch.full_like(targets,1-alpha))
    return (-a_t * (1-p_t)**gamma * torch.log(p_t.clamp(1e-7))).mean()

def train_epoch(model, loader, opt, scaler):
    model.train()
    tot,n = 0,0
    for X,y in loader:
        X,y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        with torch.autocast(DEVICE.type, torch.float16, enabled=DEVICE.type=='cuda'):
            loss = focal_loss(model(X), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tot += loss.item()*len(y); n += len(y)
    return tot/n

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    probs, labels = [], []
    for X,y in loader:
        probs.append(torch.sigmoid(model(X.to(DEVICE))).cpu())
        labels.append(y)
    probs  = torch.cat(probs).numpy()
    labels = torch.cat(labels).numpy()
    auc    = roc_auc_score(labels, probs) if labels.sum()>0 else 0.0
    return auc, probs, labels

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    steps_per_epoch=len(train_loader), epochs=30, pct_start=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type=='cuda')

EPOCHS, PATIENCE = 30, 7
best_auc, no_imp = 0.0, 0
history = {'train_loss':[], 'val_auc':[]}

print("="*60); print(f"Training PaySim model on {DEVICE}"); print("="*60)
for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, train_loader, optimizer, scaler)
    val_auc,_,_ = evaluate(model, val_loader)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_auc'].append(val_auc)
    tag = ''
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'paysim_best.pt')
        no_imp = 0; tag = ' ← best'
    else:
        no_imp += 1
    print(f"Epoch {epoch:3d}/{EPOCHS}  loss={tr_loss:.4f}  val_AUC={val_auc:.4f}{tag}")
    if no_imp >= PATIENCE:
        print(f"Early stop at epoch {epoch}")
        break
print(f"\nBest Val AUC: {best_auc:.4f}")

Training PaySim model on cuda
Epoch   1/30  loss=0.0146  val_AUC=0.9844 ← best
Epoch   2/30  loss=0.0028  val_AUC=0.9880 ← best
Epoch   3/30  loss=0.0015  val_AUC=0.9884 ← best
Epoch   4/30  loss=0.0011  val_AUC=0.9866
Epoch   5/30  loss=0.0008  val_AUC=0.9880
Epoch   6/30  loss=0.0006  val_AUC=0.9863
Epoch   7/30  loss=0.0005  val_AUC=0.9889 ← best


## 6. Test Evaluation + Optimal Threshold

In [ ]:
model.load_state_dict(torch.load('paysim_best.pt', map_location=DEVICE))
test_auc, y_prob_a, y_true_a = evaluate(model, test_loader)

print(f"Test AUC-ROC : {test_auc:.4f}")
print(f"Test AUC-PR  : {average_precision_score(y_true_a, y_prob_a):.4f}")

# Optimal threshold from F1
p_arr, r_arr, t_arr = precision_recall_curve(y_true_a, y_prob_a)
f1_arr   = 2*p_arr*r_arr/(p_arr+r_arr+1e-8)
opt_idx  = np.argmax(f1_arr)
opt_thr  = t_arr[opt_idx] if opt_idx < len(t_arr) else 0.5
y_pred_a = (y_prob_a >= opt_thr).astype(int)

print(f"\nOptimal threshold: {opt_thr:.4f}  (F1={f1_arr[opt_idx]:.4f})")
print("\n" + "="*60)
print(classification_report(y_true_a, y_pred_a,
      target_names=['Normal','Fraud'], digits=4))

print("\nThreshold sweep:")
print(f"{'Thr':<8}{'Prec':<10}{'Recall':<10}{'F1':<10}{'Caught'}")
print("-"*46)
for thr in sorted({0.2,0.3,0.4,0.5,round(float(opt_thr),2),0.6,0.7}):
    yp = (y_prob_a>=thr).astype(int)
    tp = np.sum((yp==1)&(y_true_a==1)); fp=np.sum((yp==1)&(y_true_a==0)); fn=np.sum((yp==0)&(y_true_a==1))
    p=tp/(tp+fp+1e-8); r=tp/(tp+fn+1e-8); f=2*p*r/(p+r+1e-8)
    print(f"{thr:<8.2f}{p:<10.4f}{r:<10.4f}{f:<10.4f}{tp:,}")

In [ ]:
# ── Evaluation plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# 1. Training loss curve
axes[0,0].plot(history['train_loss'], label='Train Loss', color='steelblue')
axes[0,0].set_title('Training Loss (Focal)')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss')
axes[0,0].legend(); axes[0,0].grid(True, alpha=.3)

# 2. Val AUC curve
axes[0,1].plot(history['val_auc'], label='Val AUC-ROC', color='mediumseagreen')
axes[0,1].axhline(best_auc, color='tomato', ls='--', label=f'Best={best_auc:.4f}')
axes[0,1].set_title('Validation AUC-ROC')
axes[0,1].set_xlabel('Epoch'); axes[0,1].legend(); axes[0,1].grid(True, alpha=.3)

# 3. Confusion matrix
cm = confusion_matrix(y_true_a, y_pred_a)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[0,2],
            xticklabels=['Normal','Fraud'], yticklabels=['Normal','Fraud'])
axes[0,2].set_title('Confusion Matrix')
axes[0,2].set_ylabel('Actual'); axes[0,2].set_xlabel('Predicted')

# 4. ROC curve
fpr, tpr, _ = roc_curve(y_true_a, y_prob_a)
axes[1,0].plot(fpr, tpr, lw=2, label=f'AUC={test_auc:.4f}')
axes[1,0].plot([0,1],[0,1],'k--',lw=1)
axes[1,0].set_title('ROC Curve')
axes[1,0].set_xlabel('False Positive Rate'); axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].legend(); axes[1,0].grid(True, alpha=.3)

# 5. Precision-Recall curve
ap = average_precision_score(y_true_a, y_prob_a)
axes[1,1].plot(r_arr, p_arr, lw=2, label=f'AP={ap:.4f}')
axes[1,1].scatter(r_arr[opt_idx], p_arr[opt_idx], c='red', s=100, zorder=5,
                  label=f'Opt thr={opt_thr:.3f}')
axes[1,1].set_title('Precision-Recall Curve')
axes[1,1].set_xlabel('Recall'); axes[1,1].set_ylabel('Precision')
axes[1,1].legend(); axes[1,1].grid(True, alpha=.3)

# 6. Score distribution
axes[1,2].hist(y_prob_a[y_true_a==0], bins=60, alpha=.6, label='Normal', color='steelblue', log=True)
axes[1,2].hist(y_prob_a[y_true_a==1], bins=60, alpha=.7, label='Fraud',  color='tomato',    log=True)
axes[1,2].axvline(opt_thr, color='k', ls='--', label=f'Thr={opt_thr:.3f}')
axes[1,2].set_title('Score Distribution (log y)')
axes[1,2].set_xlabel('Fraud Probability')
axes[1,2].legend(); axes[1,2].grid(True, alpha=.3)

plt.tight_layout()
plt.savefig('finguard_paysim_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved → finguard_paysim_results.png")

In [ ]:
import torch, pickle, json
import numpy as np

# ── Save model checkpoint ─────────────────────────────────────────────────────
torch.save({
    'model_state'   : model.state_dict(),
    'n_features'    : N_FEATURES,
    'feature_cols'  : FEATURE_COLS,
    'seq_len'       : SEQ_LEN,
    'opt_threshold' : float(opt_thr),
    'test_auc_roc'  : float(test_auc),
    'test_auc_pr'   : float(average_precision_score(y_true_a, y_prob_a)),
}, 'finguard_paysim.pt')

print("Saved → finguard_paysim.pt")
print(f"  AUC-ROC  : {test_auc:.4f}")
print(f"  AUC-PR   : {average_precision_score(y_true_a, y_prob_a):.4f}")
print(f"  Opt thr  : {opt_thr:.4f}")
print(f"  Features : {N_FEATURES}")

# ── Download in Google Colab ──────────────────────────────────────────────────
try:
    from google.colab import files
    files.download('finguard_paysim.pt')
    files.download('finguard_paysim_results.png')
    print("\nDownload triggered ✓")
except ImportError:
    print("\nNot running in Colab — files saved locally:")
    import os
    for f in ['finguard_paysim.pt', 'finguard_paysim_results.png']:
        if os.path.exists(f):
            print(f"  {f}  ({os.path.getsize(f)/1024:.1f} KB)")